In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/50-007-machine-learning-summer-2025/sample_submission.csv
/kaggle/input/50-007-machine-learning-summer-2025/train_tfidf_features.csv
/kaggle/input/50-007-machine-learning-summer-2025/test_tfidf_features.csv
/kaggle/input/50-007-machine-learning-summer-2025/train.csv
/kaggle/input/50-007-machine-learning-summer-2025/test.csv


**TASK 1**

In [ ]:
# Task 1: Logistic Regression Implementation from Scratch
# Hate Speech Classification - Using Pre-computed TF-IDF Features

import numpy as np
import pandas as pd
from collections import Counter
import re
import string
from typing import Tuple, List, Dict
import csv
import os

class LogisticRegression:
    """
    Logistic Regression implementation from scratch for hate speech classification.
    """
    
    def __init__(self, learning_rate: float = 0.01, max_epochs: int = 1000, 
                 tolerance: float = 1e-6, regularization: float = 0.01):
        """Initialize the Logistic Regression model."""
        self.learning_rate = learning_rate
        self.max_epochs = max_epochs
        self.tolerance = tolerance
        self.regularization = regularization
        self.weights = None
        self.bias = None
        self.loss_history = []
        
    def sigmoid(self, z: np.ndarray) -> np.ndarray:
        """Sigmoid activation function."""
        # Handle different input types
        z = np.asarray(z, dtype=np.float64)
        z = np.clip(z, -500, 500)  # Prevent overflow
        
        # Add small epsilon to prevent division issues
        exp_neg_z = np.exp(-z)
        return 1 / (1 + exp_neg_z)
    
    def loss(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        """Compute logistic loss with L2 regularization."""
        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
        ce_loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
        l2_penalty = self.regularization * np.sum(self.weights ** 2)
        return ce_loss + l2_penalty
    
    def gradients(self, X: np.ndarray, y: np.ndarray, y_pred: np.ndarray) -> Tuple[np.ndarray, float]:
        """Compute gradients for weights and bias."""
        n_samples = X.shape[0]
        error = y_pred - y
        dw = (1/n_samples) * X.T @ error + 2 * self.regularization * self.weights
        db = (1/n_samples) * np.sum(error)
        return dw, db
    
    def train(self, X: np.ndarray, y: np.ndarray, batch_size: int = 32, 
              learning_rate: float = None, epochs: int = None) -> List[float]:
        """Train the logistic regression model using gradient descent."""
        if learning_rate is not None:
            self.learning_rate = learning_rate
        if epochs is not None:
            self.max_epochs = epochs
            
        n_samples, n_features = X.shape
        
        # Initialize weights and bias
        self.weights = np.random.normal(0, 0.01, n_features)
        self.bias = 0
        
        self.loss_history = []
        prev_loss = float('inf')
        
        print(f"Training with {n_samples} samples, {n_features} features")
        
        for epoch in range(self.max_epochs):
            # Shuffle data
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            epoch_loss = 0
            n_batches = 0
            
            # Mini-batch gradient descent
            for i in range(0, n_samples, batch_size):
                end_idx = min(i + batch_size, n_samples)
                X_batch = X_shuffled[i:end_idx]
                y_batch = y_shuffled[i:end_idx]
                
                # Forward pass
                z = X_batch @ self.weights + self.bias
                
                # Check for invalid values
                if np.any(np.isnan(z)) or np.any(np.isinf(z)):
                    print("Warning: Invalid values in z, resetting weights")
                    self.weights = np.random.normal(0, 0.01, len(self.weights))
                    self.bias = 0
                    z = X_batch @ self.weights + self.bias
                
                y_pred = self.sigmoid(z)
                
                # Compute loss
                batch_loss = self.loss(y_batch, y_pred)
                epoch_loss += batch_loss
                n_batches += 1
                
                # Backward pass
                dw, db = self.gradients(X_batch, y_batch, y_pred)
                
                # Update parameters
                self.weights -= self.learning_rate * dw
                self.bias -= self.learning_rate * db
            
            avg_loss = epoch_loss / n_batches
            self.loss_history.append(avg_loss)
            
            if epoch % 50 == 0:
                print(f"Epoch {epoch}, Loss: {avg_loss:.6f}")
            
            # Check for convergence
            if abs(prev_loss - avg_loss) < self.tolerance:
                print(f"Converged at epoch {epoch}")
                break
            prev_loss = avg_loss
            
            # Early stopping if loss increases consistently
            if epoch > 100 and avg_loss > prev_loss * 1.1:
                print(f"Early stopping at epoch {epoch} due to increasing loss")
                break
        
        return self.loss_history
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Make predictions on new data."""
        if self.weights is None:
            raise ValueError("Model has not been trained yet!")
        
        z = X @ self.weights + self.bias
        probabilities = self.sigmoid(z)
        predictions = (probabilities >= 0.5).astype(int)
        return predictions
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Get prediction probabilities."""
        if self.weights is None:
            raise ValueError("Model has not been trained yet!")
        
        z = X @ self.weights + self.bias
        probabilities = self.sigmoid(z)
        return probabilities

def evaluate_model(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """Evaluate model performance."""
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

def main():
    # ============================================================================
    # LOAD ALL DATA FILES (4 FILES TOTAL) - CHANGE PATHS HERE FOR KAGGLE
    # ============================================================================
    print("Loading all data files...")
    
    try:
        # Load CSV files with text data and labels
        train_df = pd.read_csv("/kaggle/input/50-007-machine-learning-summer-2025/train.csv")      # columns: id, post, label 
        test_df = pd.read_csv("/kaggle/input/50-007-machine-learning-summer-2025/test.csv")        # columns: id, post  
        
        # Load pre-computed TF-IDF feature files
        X_train_tfidf = pd.read_csv("/kaggle/input/50-007-machine-learning-summer-2025/train_tfidf_features.csv")  # TF-IDF features for training
        X_test_tfidf = pd.read_csv("/kaggle/input/50-007-machine-learning-summer-2025/test_tfidf_features.csv")    # TF-IDF features for testing
        
        print("✓ All 4 files loaded successfully!")
        print(f"  - train.csv: {train_df.shape}")
        print(f"  - test.csv: {test_df.shape}")  
        print(f"  - train_tfidf_features.csv: {X_train_tfidf.shape}")
        print(f"  - test_tfidf_features.csv: {X_test_tfidf.shape}")
        
    except FileNotFoundError as e:
        print(f"❌ Error: Could not find required files: {e}")
        print("Make sure all 4 files are in the '/kaggle/input/50-007-machine-learning-summer-2025/' folder:")
        print("  - /kaggle/input/50-007-machine-learning-summer-2025/train.csv")
        print("  - /kaggle/input/50-007-machine-learning-summer-2025/test.csv") 
        print("  - /kaggle/input/50-007-machine-learning-summer-2025/train_tfidf_features.csv")
        print("  - /kaggle/input/50-007-machine-learning-summer-2025/test_tfidf_features.csv")
        print("\n🔧 FOR KAGGLE: Change the paths above to match your Kaggle dataset structure")
        return
    
    # ============================================================================
    # PROCESS LOADED DATA
    # ============================================================================
    
    # Extract labels and IDs
    y_train = train_df['label'].values
    test_ids = test_df['id'].tolist()
    
    print(f"\nClass distribution: {Counter(y_train)}")
    
    # Handle TF-IDF feature dimension mismatch (train has 1 extra column - the label column)
    print("\nProcessing TF-IDF features...")
    if X_train_tfidf.shape[1] == X_test_tfidf.shape[1] + 1:
        print("Detected extra column in training TF-IDF (label column), removing it...")
        # Remove the 2nd column (index 1) which contains the labels
        X_train_tfidf = X_train_tfidf.drop(X_train_tfidf.columns[1], axis=1)
        print(f"New training TF-IDF shape: {X_train_tfidf.shape}")
    
    # Final shape check
    if X_train_tfidf.shape[1] != X_test_tfidf.shape[1]:
        print(f"WARNING: Feature count mismatch!")
        print(f"Train features: {X_train_tfidf.shape[1]}, Test features: {X_test_tfidf.shape[1]}")
        # Align to minimum features
        min_features = min(X_train_tfidf.shape[1], X_test_tfidf.shape[1])
        X_train_tfidf = X_train_tfidf.iloc[:, :min_features]
        X_test_tfidf = X_test_tfidf.iloc[:, :min_features]
        print(f"Aligned both to {min_features} features")
    
    # Convert to numpy arrays for the model
    X_train = X_train_tfidf.values
    X_test = X_test_tfidf.values
    
    print(f"✓ Final TF-IDF shapes - Train: {X_train.shape}, Test: {X_test.shape}")

    # ============================================================================
    # TRAIN MODEL
    # ============================================================================
    
    # Initialize and train model
    print("\n" + "="*50)
    print("TRAINING LOGISTIC REGRESSION MODEL")
    print("="*50)
    
    # !!! Change inputs here to tune model !!!
    model = LogisticRegression(
        learning_rate=0.1,
        max_epochs=200,
        tolerance=1e-5,
        regularization=0.01
    )

    print(f"Training with {len(y_train)} samples and {X_train.shape[1]} TF-IDF features")

    # Create validation split for evaluation
    split_idx = int(0.8 * len(X_train))
    X_train_split, X_val_split = X_train[:split_idx], X_train[split_idx:]
    y_train_split, y_val_split = y_train[:split_idx], y_train[split_idx:]

    print(f"Training set: {X_train_split.shape[0]} samples")
    print(f"Validation set: {X_val_split.shape[0]} samples")

    # Train the model
    print("\nTraining model...")
    loss_history = model.train(X_train_split, y_train_split, batch_size=64)

    # ============================================================================
    # EVALUATE MODEL
    # ============================================================================
    
    print("\n" + "="*50)
    print("EVALUATING MODEL PERFORMANCE")
    print("="*50)
    
    # Make predictions
    train_pred = model.predict(X_train_split)
    val_pred = model.predict(X_val_split)

    # Evaluate performance
    train_metrics = evaluate_model(y_train_split, train_pred)
    val_metrics = evaluate_model(y_val_split, val_pred)

    print("\n=== Training Performance ===")
    for metric, value in train_metrics.items():
        print(f"{metric}: {value:.4f}")

    print("\n=== Validation Performance ===")
    for metric, value in val_metrics.items():
        print(f"{metric}: {value:.4f}")

    # ============================================================================
    # GENERATE FINAL PREDICTIONS
    # ============================================================================
    
    print("\n" + "="*60)
    print("GENERATING FINAL PREDICTIONS FOR TEST SET")
    print("="*60)

    # Retrain on full training data for final model
    print("Retraining on full training data for final predictions...")
    final_model = LogisticRegression(
        learning_rate=0.1,
        max_epochs=200,
        tolerance=1e-5,
        regularization=0.01
    )
    
    final_model.train(X_train, y_train, batch_size=64)

    # Make predictions on test set
    test_pred = final_model.predict(X_test)
    test_probabilities = final_model.predict_proba(X_test)

    print(f"Generated predictions for {len(test_pred)} test samples")
    print(f"Prediction distribution: {Counter(test_pred)}")

    # ============================================================================
    # SAVE PREDICTIONS
    # ============================================================================
    
    print("\nSaving predictions to LogReg_Prediction.csv...")

    submission_data = []
    for i, (test_id, prob, pred) in enumerate(zip(test_ids, test_probabilities, test_pred)):
        submission_data.append({
            'row ID': test_id,
            'label': int(pred),
        })

    # Save to results folder
    os.makedirs('results', exist_ok=True)
    output_path = os.path.join('results', 'LogReg_Prediction.csv')
    
    with open(output_path, 'w', newline='') as csvfile:
        fieldnames = ['row ID', 'label', 'probability']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(submission_data)

    print(f"✓ Task 1 Complete! LogReg_Prediction.csv saved to {os.path.abspath(output_path)}")
    print(f"✓ Final validation accuracy: {val_metrics['accuracy']:.4f}")
    print(f"✓ Final validation F1-score: {val_metrics['f1_score']:.4f}")

if __name__ == "__main__":
    main()

**TASK 2**

In [ ]:
# Task 2: PCA + k-NN Implementation from Scratch
# Dimensionality Reduction + Classification

import numpy as np
import pandas as pd
from typing import Tuple, List, Dict
import csv
import os
from collections import Counter

class PCA:
    """
    Principal Component Analysis (PCA) implementation from scratch.
    Reduces dimensionality by finding the most important features in the data.
    """
    
    def __init__(self, n_components: int = None):
        """Initialize PCA with specified number of components to retain."""
        self.n_components = n_components          # Number of components to keep
        self.components = None                    # Principal component vectors
        self.mean = None                         # Mean of training data
        self.explained_variance_ratio = None     # Variance explained by each component
        self.eigenvalues = None                  # Eigenvalues from decomposition
        
    def fit(self, X: np.ndarray) -> 'PCA':
        """
        Fit PCA model to training data by computing principal components.
        
        Args:
            X: Training data matrix of shape (n_samples, n_features)
        
        Returns:
            self: Fitted PCA instance
        """
        # Step 1: Center the data by subtracting mean
        self.mean = np.mean(X, axis=0)
        X_centered = X - self.mean
        
        # Step 2: Compute covariance matrix
        cov_matrix = np.cov(X_centered.T)
        
        # Step 3: Perform eigendecomposition
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        
        # Step 4: Sort eigenvalues and eigenvectors in descending order
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]
        
        # Step 5: Calculate explained variance ratios
        self.eigenvalues = eigenvalues
        total_variance = np.sum(eigenvalues)
        self.explained_variance_ratio = eigenvalues / total_variance
        
        # Step 6: Determine number of components if not specified
        if self.n_components is None:
            # Automatically select components that explain 95% of variance
            cumsum_variance = np.cumsum(self.explained_variance_ratio)
            self.n_components = np.argmax(cumsum_variance >= 0.95) + 1
            print(f"Auto-selected {self.n_components} components (95% variance threshold)")
        
        # Check if requested components exceed available dimensions
        max_components = min(X.shape[0], X.shape[1])
        if self.n_components > max_components:
            print(f"Warning: Requested {self.n_components} components, but maximum available is {max_components}")
            self.n_components = max_components
        
        # Step 7: Store selected principal components
        self.components = eigenvectors[:, :self.n_components]
        
        print(f"PCA fitted with {self.n_components} components")
        print(f"Total variance explained: {np.sum(self.explained_variance_ratio[:self.n_components]):.4f}")
        
        return self
        
    def transform(self, X: np.ndarray) -> np.ndarray:
        """
        Transform data to reduced dimensional space using fitted components.
        
        Args:
            X: Data to transform of shape (n_samples, n_features)
            
        Returns:
            X_transformed: Transformed data of shape (n_samples, n_components)
        """
        if self.components is None:
            raise ValueError("PCA must be fitted before transformation. Call fit() first.")
            
        # Center the data using training mean
        X_centered = X - self.mean
        
        # Project data onto principal components
        X_transformed = X_centered @ self.components
        
        return X_transformed
    
    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        """Fit PCA model and transform data in one step."""
        return self.fit(X).transform(X)

class KNN:
    """
    k-Nearest Neighbors (k-NN) classifier implementation from scratch.
    Makes predictions based on majority vote of k nearest training samples.
    """
    
    def __init__(self, k: int = 5, distance_metric: str = 'euclidean'):
        """
        Initialize k-NN classifier.
        
        Args:
            k: Number of nearest neighbors to consider for prediction
            distance_metric: Distance metric ('euclidean' or 'manhattan')
        """
        self.k = k
        self.distance_metric = distance_metric
        self.X_train = None     # Training feature vectors
        self.y_train = None     # Training labels
        
    def fit(self, X: np.ndarray, y: np.ndarray) -> 'KNN':
        """
        Fit k-NN model by storing training data.
        
        Args:
            X: Training features of shape (n_samples, n_features)
            y: Training labels of shape (n_samples,)
            
        Returns:
            self: Fitted k-NN instance
        """
        self.X_train = X.copy()
        self.y_train = y.copy()
        print(f"k-NN fitted with {len(X)} training samples, k={self.k}")
        return self
    
    def _calculate_distance(self, x1: np.ndarray, x2: np.ndarray) -> float:
        """
        Calculate distance between two feature vectors.
        
        Args:
            x1, x2: Feature vectors to compare
            
        Returns:
            distance: Computed distance value
        """
        if self.distance_metric == 'euclidean':
            return np.sqrt(np.sum((x1 - x2) ** 2))
        elif self.distance_metric == 'manhattan':
            return np.sum(np.abs(x1 - x2))
        else:
            raise ValueError(f"Unsupported distance metric: {self.distance_metric}")
    
    def _get_neighbors(self, x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Find k nearest neighbors for a query point.
        
        Args:
            x: Query point feature vector
            
        Returns:
            neighbor_indices: Indices of k nearest neighbors
            distances: Distances to k nearest neighbors
        """
        # Calculate distances to all training points
        distances = []
        for i, x_train in enumerate(self.X_train):
            dist = self._calculate_distance(x, x_train)
            distances.append((dist, i))
        
        # Sort by distance and select k nearest
        distances.sort(key=lambda x: x[0])
        k_nearest = distances[:self.k]
        
        neighbor_indices = [idx for _, idx in k_nearest]
        neighbor_distances = [dist for dist, _ in k_nearest]
        
        return np.array(neighbor_indices), np.array(neighbor_distances)
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Predict class labels using majority vote of k nearest neighbors.
        
        Args:
            X: Test features of shape (n_samples, n_features)
            
        Returns:
            predictions: Predicted class labels of shape (n_samples,)
        """
        if self.X_train is None:
            raise ValueError("Model must be fitted before prediction. Call fit() first.")
        
        predictions = []
        
        for i, x in enumerate(X):
            if i % 100 == 0:
                print(f"Processing sample {i}/{len(X)}")
                
            # Find k nearest neighbors
            neighbor_indices, _ = self._get_neighbors(x)
            
            # Get labels of nearest neighbors
            neighbor_labels = self.y_train[neighbor_indices]
            
            # Majority vote for prediction
            label_counts = Counter(neighbor_labels)
            predicted_label = label_counts.most_common(1)[0][0]
            
            predictions.append(predicted_label)
        
        return np.array(predictions)

def load_and_preprocess_data(train_path: str, test_path: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Load and preprocess training and test datasets.
    
    Args:
        train_path: Path to training data CSV file
        test_path: Path to test data CSV file
        
    Returns:
        X_train: Training feature matrix
        y_train: Training labels
        X_test: Test feature matrix
    """
    print("Loading training data...")
    
    # Load training dataset
    df_train = pd.read_csv(train_path)
    
    # Validate data format
    if 'post' in df_train.columns:
        print("Error: Raw text data detected. This implementation requires TF-IDF features.")
        print("Please ensure you are using the correct preprocessed feature files.")
        return None, None, None
    
    # Separate features from labels
    feature_columns = [col for col in df_train.columns if col != 'label']
    X_train = df_train[feature_columns].values
    y_train = df_train['label'].values
    
    print(f"Training data loaded: {X_train.shape[0]} samples, {X_train.shape[1]} features")
    
    # Load test data
    print("Loading test data...")
    df_test = pd.read_csv(test_path)
    X_test = df_test[feature_columns].values
    print(f"Test data loaded: {X_test.shape[0]} samples")
    
    return X_train, y_train, X_test

def evaluate_model(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """
    Evaluate binary classification model performance.
    
    Args:
        y_true: True class labels
        y_pred: Predicted class labels
        
    Returns:
        metrics: Dictionary containing accuracy, precision, recall, and F1-score
    """
    # Calculate confusion matrix components
    true_positive = np.sum((y_true == 1) & (y_pred == 1))
    true_negative = np.sum((y_true == 0) & (y_pred == 0))
    false_positive = np.sum((y_true == 0) & (y_pred == 1))
    false_negative = np.sum((y_true == 1) & (y_pred == 0))
    
    # Calculate performance metrics
    accuracy = (true_positive + true_negative) / len(y_true)
    precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
    recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1_score
    }

def main():
    """
    Main execution function for PCA + k-NN pipeline.
    Creates prediction files for all required PCA component numbers.
    """
    
    # Define data file paths
    train_tfidf_path = os.path.join('/kaggle/input/50-007-machine-learning-summer-2025', 'train_tfidf_features.csv')
    test_tfidf_path = os.path.join('/kaggle/input/50-007-machine-learning-summer-2025', 'test_tfidf_features.csv')
    
    # Load and preprocess data
    X_train, y_train, X_test = load_and_preprocess_data(train_tfidf_path, test_tfidf_path)
    
    if X_train is None:
        print("Data loading failed. Please verify file paths and data format.")
        return
    
    print(f"\nInitial feature space: {X_train.shape[1]} dimensions")
    print("Processing all required PCA component numbers...")
    
    # Assignment requires testing these specific component numbers
    n_components_options = [2000, 1000, 500, 100]
    k = 2  # Assignment specifies n_neighbors=2
    
    # Create results directory
    os.makedirs('results', exist_ok=True)
    
    # Process each required component number
    for n_comp in n_components_options:
        print(f"\n{'='*60}")
        print(f"Processing PCA with {n_comp} components and k={k}")
        print(f"{'='*60}")
        
        try:
            # Apply PCA transformation
            pca = PCA(n_components=n_comp)
            X_train_pca = pca.fit_transform(X_train)
            X_test_pca = pca.transform(X_test)
            
            # Train k-NN classifier
            knn = KNN(k=k, distance_metric='euclidean')
            knn.fit(X_train_pca, y_train)
            
            # Generate predictions for test set
            print(f"\nGenerating test predictions...")
            test_predictions = knn.predict(X_test_pca)
            
            # Create submission file in required format
            print(f"Creating submission file...")
            
            submission_data = []
            for i, pred in enumerate(test_predictions):
                submission_data.append({
                    'id': i,
                    'label': int(pred)
                })
            
            # Save to CSV file
            output_path = os.path.join('results', f'PCA_KNN_Prediction_{n_comp}.csv')
            
            with open(output_path, 'w', newline='') as csvfile:
                fieldnames = ['id', 'label']
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(submission_data)
            
            print(f"Predictions saved to: {output_path}")
            print(f"Prediction distribution: {Counter(test_predictions)}")
            
        except Exception as e:
            print(f"Error processing {n_comp} components: {str(e)}")
            continue
    
    print(f"\n{'='*60}")
    print("All prediction files generated successfully!")
    print("Files created:")
    for n_comp in n_components_options:
        print(f"  - PCA_KNN_Prediction_{n_comp}.csv")
    print("These files can now be submitted to Kaggle to obtain Macro F1 scores.")
    print(f"All models use k-NN with k={k} as required by the assignment.")

if __name__ == "__main__":
    main()

**TASK 3** - Here we are using a SVM model, because we believe its the best fit for a high-dimensional data like text features

In [ ]:
import numpy as np
import pandas as pd
import os
from collections import Counter
import math

# Load pre-computed TF-IDF features
print("Loading pre-computed TF-IDF features...")
train_tfidf_df = pd.read_csv("/kaggle/input/50-007-machine-learning-summer-2025/train_tfidf_features.csv")
test_tfidf_df = pd.read_csv("/kaggle/input/50-007-machine-learning-summer-2025/test_tfidf_features.csv")

print(f"Loaded {len(train_tfidf_df)} training samples and {len(test_tfidf_df)} test samples")

# Extract labels and features
y_train = train_tfidf_df['label'].values
test_ids = test_tfidf_df['id'].tolist()

# Get feature columns (all columns except 'id' and 'label' for train, except 'id' for test)
train_feature_cols = [col for col in train_tfidf_df.columns if col not in ['id', 'label']]
test_feature_cols = [col for col in test_tfidf_df.columns if col != 'id']

X_train = train_tfidf_df[train_feature_cols].values
X_test = test_tfidf_df[test_feature_cols].values

print(f"Feature matrix shape: {X_train.shape}")
print(f"Class distribution: {Counter(y_train)}")

# Convert original labels to -1 and 1 for SVM
y_train = np.where(y_train == 0, -1, 1)
print(f"SVM labels: {np.unique(y_train, return_counts=True)}")

# Add bias term to both train and test
X_train = np.hstack([X_train, np.ones((X_train.shape[0], 1))])
X_test = np.hstack([X_test, np.ones((X_test.shape[0], 1))])


# Enhanced SVM hyperparameters with class balancing

# !!! Change values here to tune model !!!

print("\nEnhanced SVM Configuration:")
learning_rate = 0.005   # Reduced for more stable training
lambda_param = 0.0001   # Lower regularization for complex features
n_iters = 600           # Balanced iterations for convergence

# Class balancing - give more weight to minority class
class_counts = Counter(y_train)
class_weights = {
    -1: len(y_train) / (2 * class_counts[-1]),  # Weight for class 0 (hate speech)
    1: len(y_train) / (2 * class_counts[1])     # Weight for class 1 (normal)
}

print(f"Learning rate: {learning_rate}")
print(f"Regularization: {lambda_param}")
print(f"Iterations: {n_iters}")
print(f"Class weights: {class_weights}")

# Initialize weights with Xavier/Glorot initialization
fan_in = X_train.shape[1]
w = np.random.normal(0, math.sqrt(2.0 / fan_in), X_train.shape[1])
print(f"Initialized weights shape: {w.shape}")
print(f"Weight initialization std: {math.sqrt(2.0 / fan_in):.6f}")

# Advanced Training with Stochastic Sub-Gradient Descent + Class Balancing
print("\nTraining Enhanced SVM with Class Balancing...")

# Learning rate scheduling
def get_learning_rate(epoch, initial_lr):
    # Decay learning rate after certain epochs
    if epoch < 200:
        return initial_lr
    elif epoch < 500:
        return initial_lr * 0.5
    else:
        return initial_lr * 0.1

for epoch in range(n_iters):
    total_loss = 0
    current_lr = get_learning_rate(epoch, learning_rate)
    
    # Shuffle training data for better convergence
    indices = np.random.permutation(len(X_train))
    
    for idx in indices:
        x_i = X_train[idx]
        y_i = y_train[idx]
        
        condition = y_i * np.dot(x_i, w)
        class_weight = class_weights[y_i]
        
        if condition >= 1:
            grad = 2 * lambda_param * w
            loss = 0
        else:
            # Apply class weighting to the gradient
            grad = 2 * lambda_param * w - class_weight * y_i * x_i
            loss = class_weight * (1 - condition)
        
        w = w - current_lr * grad
        total_loss += loss
    
    # Enhanced logging
    if (epoch + 1) % 100 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1}/{n_iters} - Loss: {total_loss:.4f} - LR: {current_lr:.6f} - ||w||: {np.linalg.norm(w):.4f}")

print("Enhanced SVM training completed!")

# Enhanced prediction function with confidence scores
def predict(X, return_confidence=False):
    scores = np.dot(X, w)
    predictions = np.where(scores >= 0, 1, 0)  # return 0/1 labels
    
    if return_confidence:
        # Convert scores to confidence (distance from decision boundary)
        confidence = np.abs(scores)
        return predictions, confidence
    return predictions

def predict_proba(X):
    """Predict class probabilities using sigmoid function."""
    scores = np.dot(X, w)
    # Apply sigmoid to convert to probabilities
    probabilities = 1 / (1 + np.exp(-scores))
    return np.column_stack([1 - probabilities, probabilities])

# Enhanced training performance evaluation
y_train_pred, train_confidence = predict(X_train, return_confidence=True)
train_acc = np.mean(y_train_pred == (y_train == 1).astype(int))

# Calculate precision, recall, F1 for training
y_train_binary = (y_train == 1).astype(int)
tp = np.sum((y_train_binary == 1) & (y_train_pred == 1))
fp = np.sum((y_train_binary == 0) & (y_train_pred == 1))
fn = np.sum((y_train_binary == 1) & (y_train_pred == 0))

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Training Performance:")
print(f"  Accuracy: {train_acc:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")
print(f"  F1-Score: {f1:.4f}")
print(f"  Predictions distribution: {Counter(y_train_pred)}")
print(f"  Average confidence: {np.mean(train_confidence):.4f}")

# Enhanced test prediction with confidence analysis
y_test_pred, test_confidence = predict(X_test, return_confidence=True)
test_probabilities = predict_proba(X_test)

print(f"Test Performance:")
print(f"  Predictions distribution: {Counter(y_test_pred)}")
print(f"  Average confidence: {np.mean(test_confidence):.4f}")
print(f"  High confidence predictions (>1.0): {np.sum(test_confidence > 1.0)}")
print(f"  Low confidence predictions (<0.5): {np.sum(test_confidence < 0.5)}")

# Save predictions with proper format
print("Saving predictions...")
os.makedirs('results', exist_ok=True)

submission_data = []
for test_id, pred in zip(test_ids, y_test_pred):
    submission_data.append({
        'row ID': test_id,
        'label': int(pred)
    })

import csv
output_path = os.path.join('results', 'Task3_SVM_Prediction.csv')
with open(output_path, 'w', newline='') as csvfile:
    fieldnames = ['row ID', 'label']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(submission_data)

print(f"Task 3 Enhanced SVM Complete! Predictions saved to {os.path.abspath(output_path)}")
print(f"Generated {len(y_test_pred)} predictions with pre-computed TF-IDF features")
print(f"Model improvements:")
print(f"  - Using pre-computed TF-IDF features with {X_train.shape[1]} features")
print(f"  - Class-balanced training with weighted loss")
print(f"  - Learning rate scheduling")
print(f"  - Xavier/Glorot weight initialization for better convergence")
